**Imports**

In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from torchvision.models.vision_transformer import vit_b_16
from torchvision.models import ViT_B_16_Weights
from torchvision.datasets import ImageFolder
from torch.utils.data import Sampler
import os
import random
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
!unzip /content/drive/MyDrive/AML-Project/market.zip

**ViTVAE definition**

In [ ]:
#### ViTvae ####

class MyVIT(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    self.preprocessing = ViT_B_16_Weights.DEFAULT.transforms()

  def freezeFirst11Layers(self):
    for layer in self.vit.encoder.layers[-12:-1]:
        print(layer.__class__.__name__)
        for i in layer.children():
            for p in i.parameters():
                p.requires_grad = False

    for i in self.vit.conv_proj.parameters():
      i.requires_grad = False

    self.vit.encoder.pos_embedding.requires_grad = False

    for i in self.vit.heads.head.parameters():
      i.requires_grad = False



  def forward(self, x):
    x = self.vit._process_input(x)
    batch_class_token = self.vit.class_token.expand(x.shape[0], -1, -1)
    feats = torch.cat([batch_class_token, x], dim=1)
    feats = self.vit.encoder(feats) # [197,768] (151296), devo estrarre [196,768]
    global_token = feats[:,0,:]
    local_tokens = feats[:,1:,:]
    TOT_tokens = local_tokens + global_token.unsqueeze(1)
    TOT_tokens = TOT_tokens.reshape((-1,14,14,768))
    return TOT_tokens #output shape = [b,14,14,768]

class VitVAE(torch.nn.Module):
  def __init__(self, latent_dim = 256):
    super().__init__()

    self.vit = MyVIT()
    self.latent_dim = latent_dim
    self.conv_layer = nn.Conv2d(in_channels=768, out_channels=64, kernel_size=3, stride=1, padding=1)
    self.conv_layer2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, stride=1, padding=1)
    self.maxPool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)
    self.linear = nn.Linear(32*7*7, latent_dim)

    self.mu = nn.Linear(self.latent_dim, self.latent_dim)
    self.log_var = nn.Linear(self.latent_dim, self.latent_dim)

    self.linear2 = nn.Linear(latent_dim,32*7*7)
    self.deconv_layer = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
    self.deconv_layer1 = nn.Conv2d(in_channels=64, out_channels=768, kernel_size=3, stride=1, padding=1)
    self.maxUnPool = nn.MaxUnpool2d(kernel_size=2, stride=2)
    self.relu = nn.ReLU()

  def freezeFirst11Layers(self):
    self.vit.freezeFirst11Layers()

  def reparameterize(self, mu, log_var):
    # Compute the standard deviation from the log variance
    std = torch.exp(0.5 * log_var)
    # Generate random noise using the same shape as std
    eps = torch.randn_like(std)
    # Return the reparameterized sample
    return mu + eps * std

  def forward(self,x):
    tokens = self.vit(x) # [b,14,14,768]
    tokens = tokens.permute(0,3,1,2) # [b,14,14,768] -> [b,768,14,14]

    ##### ENCODER #####

    x = self.relu(self.conv_layer(tokens)) # Shape: [b,64,14,14]
    x = self.relu(self.conv_layer2(x)) # Shape: [b,32,14,14]
    x,inds = self.maxPool(x) # Shape: [b,32,7,7]

    x = torch.flatten(x, start_dim=1) #Shape: [b,32*7*7 = 1568]
    x = self.relu(self.linear(x)) #Shape: [b,latent_dim]

    mu = self.mu(x)
    log_var = self.log_var(x)

    z = self.reparameterize(mu,log_var) #latent codes [b,latent_dim]

    ##### DECODER #####

    x = self.relu(self.linear2(z)) #Shape: [b,32,7,7]
    x = x.reshape((-1,32,7,7))
    x = self.maxUnPool(x,inds) #Shape: [32,14,14]
    x = self.relu(self.deconv_layer(x)) #Shape: [64,14,14]
    x = self.relu(self.deconv_layer1(x)) #Shape: [178,14,14]
    return mu, log_var, x, z, tokens

  def VAEloss(self,reconstructed_x,x,mu,log_var):
    x = x.reshape((-1,14,14,768)).permute((0,3,1,2))
    BCE = F.mse_loss(reconstructed_x, x, reduction='mean')
    # Compute the Kullback-Leibler divergence between the learned latent variable distribution and a standard Gaussian distribution
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    KLD /= x.size(0)
    # Combine the two losses by adding them together and return the result
    return BCE + KLD


**Sampler** we need the sampler to create batches made of anchor, positive and negativs

In [ ]:
#### SAMPLER ####


t = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Resize((224,224)),       # Resize images to 224 x 224 volendo si potrebbe fare ma ci mette troppo tempo (su a100 si)
])

marketDS_TRAIN = ImageFolder('/content/Market/train', transform=t)


class tripletSampler(Sampler): # IMPORTANTE: togliere dal ds le classi che hanno 1 solo elemento
  def __init__(self, data, batch_size,n):
    # build data for sampling here
    self.batch_size = batch_size
    self.classes = [i for i in range(750)]
    self.n = n #numero di classi secondarie
    self.data = list(data)
    #self.flatDS = [j for i in a for j in data[i]] #lista flattata

    self.indexedIndexes = {}
    for i in range(len(self.data)):
      self.indexedIndexes[data[i][1]] = []
    for i in range(len(self.data)):
      self.indexedIndexes[data[i][1]].append(i)


  def __iter__(self):
    for i in [35, 62, 728]:
      if i in self.classes:
        self.classes.remove(i)

    for classs in self.classes:
      batchIndxs = []
      for _ in range(classs,classs + self.batch_size):
        secondaryClasses = []
        while len(secondaryClasses) < self.n:
          rc = random.choice(self.classes)
          if rc != classs and rc not in secondaryClasses:
            secondaryClasses.append(rc)

        primaryInd= random.sample(self.indexedIndexes[classs],2)
        secondaryInd = []
        for i in secondaryClasses:
          secondaryInd.append(random.choice(self.indexedIndexes[i]))
        batchIndxs = batchIndxs + primaryInd + secondaryInd

      yield batchIndxs #ritorna gli indcidi di tutti i batch di tutti gli elementi flattati (n_el * batch_size)


  def __len__(self):
      return len(self.data)

**Train loop**

In [ ]:
import json
import time
device = "cuda" if torch.cuda.is_available() else "cpu"

def countPar(model):
  total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  return total_trainable_params

def train():
  n_negatives = 5
  bs = 16
  epochs = 20

  vaeVit = VitVAE()
  #vaeVit.load_state_dict(torch.load(f'/content/drive/MyDrive/PPPP/ep9.pt', map_location=device)) # pretrained model
  vaeVit.freezeFirst11Layers()

  #print(f'numero parametri = {countPar(vaeVit)}')
  vaeVit.to(device)
  optimizer = torch.optim.AdamW(vaeVit.parameters(), lr=1e-3)

  train_loader = DataLoader(dataset = marketDS_TRAIN, batch_sampler = tripletSampler(marketDS_TRAIN, bs, n_negatives))
  print('train_loader loaded')

  t = transforms.Resize((224 , 224))

  with tqdm(total=epochs, desc='Traning progress', unit='epoch') as epoch_pbar:
    #FILE_LOSS = open('/content/drive/MyDrive/PPPP/LOSS_LOCAL.json', 'a')
    for ep in range(epochs):
      vaeVit.train()
      print(f'epoca {ep}')
      a = time.time()
      k = 0
      avgRecLoss = 0
      avgPosLoss = 0
      avgNegLoss = 0
      avgTotLoss = 0
      for el in train_loader:
        k +=1
        imgs, targets = el #batch di immagini e targets (bs*(n_negatives+2), 3, 128, 64) es. [112, 3, 128, 64]
        imgs = imgs.to(device)
        targets = targets.to(device)

        resized_batch = t(imgs)
        mu, log_var, x, z, tokens = vaeVit(resized_batch) # x shape = torch.Size([8, 768, 14, 14]) z shape = torch.Size([8, 256]) tokens shape = torch.Size([8, 768, 14, 14])

        recLoss = vaeVit.VAEloss(x,tokens,mu,log_var) * 100 #reconstruction loss LEVARE * 100 #######

        z = z.reshape((bs,n_negatives+2,256)) # [bs,n_negatives+2,256]  es. [16,7,256]

        positiveLoss = F.mse_loss(z[:,0,:],z[:,1,:]) *2

        negativeLoss = torch.tensor(0, dtype=torch.float32)
        for i in range(z.shape[1]-2):
            negativeLoss = negativeLoss + F.mse_loss(z[:,0,:],z[:,i+2,:]) + F.mse_loss(z[:,1,:],z[:,i+2,:])

        avgRecLoss += recLoss.item()
        avgPosLoss += positiveLoss.item()
        avgNegLoss += negativeLoss.item()
        totloss = positiveLoss - negativeLoss/(n_negatives*bs)#+recLoss
        avgTotLoss += totloss.item()

        epoch_pbar.set_postfix({
                "RecLossAVG": f"{avgRecLoss/k:.4f}",
                "PosLossAVG": f"{avgPosLoss/k:.4f}",
                "NegLossAVG": f"{avgNegLoss/(k*n_negatives*bs):.4f}",
                "TotLoss": f"{totloss:.4f}",
                "TotLossAVG": f"{avgTotLoss/k:.4f}",
            })

        #print(f'totloss = {totloss}, recLoss = {recLoss}, positiveLoss = {positiveLoss}, negativeLoss = {negativeLoss/(n_negatives*bs)}')
        #FILE_LOSS.write(json.dumps({"recLoss":recLoss.item(),"negativeLoss":negativeLoss.item()/(n_negatives*bs),"positiveLoss":positiveLoss.item(),"totloss":totloss.item(),"avgRecLoss":avgRecLoss,"avgPosLoss":avgPosLoss,"avgNegLoss":avgNegLoss}) + '\n')
        totloss.backward()
        optimizer.step()
        optimizer.zero_grad()


      torch.save(vaeVit.state_dict(), f'/content/drive/MyDrive/PPPP/FTep{ep}.pt')
      with open('/content/drive/MyDrive/PPPP/LOSS_FT.json', 'a') as file:
        file.write(json.dumps({"totloss":totloss.item(),"avgRecLoss":avgRecLoss,"avgPosLoss":avgPosLoss,"avgNegLoss":avgNegLoss}) + '\n')
        #file.write(json.dumps({"totloss":totloss.item()/k,"avgPosLoss":avgPosLoss/k,"avgNegLoss":avgNegLoss/(k*n_negatives*bs)}) + '\n')
      print(f'c ho messo {time.time()-a} secondi')
      print(f'totloss = {totloss}, recLoss = {recLoss}, positiveLoss = {positiveLoss}, negativeLoss = {negativeLoss/(n_negatives*bs)}')

train()


#ho moltiplicato la recLoss *100 dato che due vettori per essere simili questa deve essere MOLTO piccola, in questo modo si specializza prima nel ricostruire e poi nel posizionare gli embeddings nel modo giusto
